# Task 3 — Portfolio Construction (Phase 3)
**QFI Internship — Team 2 (Market Risk)**
**Submitted by: Atharv Pratap Singh Chauhan**

**Goal of this task:** turn the cleaned returns from Task 2 into an actual invested
portfolio and track its money value over time.

Inputs:
- Tickers and weights (equal-weighted).
- Initial portfolio value.
- Rebalancing frequency.

Outputs:
- Daily portfolio **value**.
- Daily portfolio **P&L** (profit and loss).
- Daily portfolio **returns**.
- Supporting charts: portfolio value, drawdown, return distribution.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
print('Libraries loaded.')

## 1. Load cleaned data from Task 2
We load the clean price matrix and the clean return matrix produced in Task 2.
If those are not present yet, we fall back and rebuild returns from the raw prices,
so this notebook always runs on its own.

In [ ]:
import os

if os.path.exists('Clean_Price_Matrix.csv') and os.path.exists('Clean_Return_Matrix.csv'):
    prices  = pd.read_csv('Clean_Price_Matrix.csv',  index_col=0, parse_dates=True)
    returns = pd.read_csv('Clean_Return_Matrix.csv', index_col=0, parse_dates=True)
    print('Loaded cleaned data from Task 2.')
else:
    # Fallback: rebuild from the originally shared raw prices
    src = 'Raw_Portfolio_Price.csv' if os.path.exists('Raw_Portfolio_Price.csv') else '00003070-Raw_Portfolio_Price.csv'
    prices  = pd.read_csv(src, index_col=0, parse_dates=True).ffill().dropna()
    returns = np.log(prices / prices.shift(1)).dropna()
    print('Task 2 outputs not found - rebuilt returns from raw prices (fallback).')

tickers = list(prices.columns)
print(f'{len(returns)} return days x {len(tickers)} stocks')
returns.head()

## 2. Portfolio inputs
We set the construction choices here so they are easy to change:
- **Weights:** equal-weighted (`1/N` each).
- **Initial value:** the starting amount invested.
- **Rebalancing:** how often we reset back to target weights.

In [ ]:
INITIAL_VALUE = 1_000_000        # starting portfolio value (currency units)
REBALANCE     = 'ME'             # 'ME' = month-end, 'QE' = quarter-end, None = buy & hold
n_assets      = len(tickers)

weights = pd.Series(1.0 / n_assets, index=tickers)   # equal weights
print('Initial portfolio value :', f'{INITIAL_VALUE:,.0f}')
print('Rebalancing frequency   :', REBALANCE)
print('Weight per stock        :', round(weights.iloc[0], 4), '| sum =', round(weights.sum(), 6))

## 3. Daily portfolio returns
The portfolio return each day is the weighted average of the individual stock returns.

We support **rebalancing**: at each rebalance date the weights are reset to the equal-weight
target. Between rebalances weights drift with the individual stocks (buy-and-hold behaviour).

In [ ]:
# Simple weighted daily return with equal weights held constant (target weights)
portfolio_returns = returns.mul(weights, axis=1).sum(axis=1)
portfolio_returns.name = 'Portfolio'

# --- Rebalancing-aware version -------------------------------------------------
# Between rebalance dates, each stock's weight drifts with its own return; on a
# rebalance date we reset back to equal weights.
if REBALANCE:
    rebal_dates = returns.resample(REBALANCE).last().index
    cur_w = weights.copy()
    port_ret = []
    for date, row in returns.iterrows():
        # today's portfolio return using yesterday's (drifted) weights
        r = float((cur_w * row).sum())
        port_ret.append(r)
        # drift the weights by today's returns
        cur_w = cur_w * np.exp(row)
        cur_w = cur_w / cur_w.sum()
        # reset to target on rebalance dates
        if date in set(rebal_dates):
            cur_w = weights.copy()
    portfolio_returns = pd.Series(port_ret, index=returns.index, name='Portfolio')

portfolio_returns.head()

## 4. Daily portfolio value and P&L
- **Value:** start at the initial amount and grow it by the daily returns (compounded).
- **P&L:** the day-over-day change in portfolio value (profit or loss in currency units).

In [ ]:
# Portfolio value = initial * cumulative growth
portfolio_value = INITIAL_VALUE * (1 + portfolio_returns).cumprod()
portfolio_value.name = 'Portfolio_Value'

# Add the starting point (day 0) so the first P&L is measured from the initial value
start_day = portfolio_value.index[0] - pd.Timedelta(days=1)
portfolio_value_full = pd.concat([pd.Series({start_day: INITIAL_VALUE}), portfolio_value])

# Daily P&L = change in value
portfolio_pnl = portfolio_value_full.diff().dropna()
portfolio_pnl.name = 'Portfolio_PnL'

result = pd.DataFrame({
    'Return': portfolio_returns,
    'Value' : portfolio_value,
    'PnL'   : portfolio_pnl,
})
result.head()

In [ ]:
# Headline numbers
total_return = portfolio_value.iloc[-1] / INITIAL_VALUE - 1
years        = (portfolio_value.index[-1] - portfolio_value.index[0]).days / 365.25
cagr         = (portfolio_value.iloc[-1] / INITIAL_VALUE) ** (1 / years) - 1
ann_vol      = portfolio_returns.std() * np.sqrt(252)
sharpe       = (portfolio_returns.mean() * 252) / ann_vol

print(f'Final portfolio value : {portfolio_value.iloc[-1]:,.0f}')
print(f'Total return          : {total_return:6.2%}')
print(f'CAGR                  : {cagr:6.2%}')
print(f'Annualized volatility : {ann_vol:6.2%}')
print(f'Sharpe (rf=0)         : {sharpe:6.2f}')

## 5. Charts
### 5a. Portfolio value over time

In [ ]:
plt.figure(figsize=(12, 5))
portfolio_value.plot(color='navy')
plt.title('Daily Portfolio Value')
plt.ylabel('Portfolio value')
plt.tight_layout()
plt.show()

### 5b. Daily P&L

In [ ]:
plt.figure(figsize=(12, 4))
colors = ['green' if x >= 0 else 'red' for x in portfolio_pnl]
plt.bar(portfolio_pnl.index, portfolio_pnl.values, color=colors, width=1.0)
plt.title('Daily Portfolio P&L')
plt.ylabel('Profit / Loss')
plt.tight_layout()
plt.show()

### 5c. Drawdown
Drawdown measures how far the portfolio is below its previous peak — a key risk view.

In [ ]:
running_max = portfolio_value.cummax()
drawdown    = portfolio_value / running_max - 1

plt.figure(figsize=(12, 4))
drawdown.plot(color='darkred')
plt.fill_between(drawdown.index, drawdown.values, 0, color='darkred', alpha=0.3)
plt.title('Portfolio Drawdown')
plt.ylabel('Drawdown')
plt.tight_layout()
plt.show()

print('Maximum drawdown :', f'{drawdown.min():.2%}')

### 5d. Distribution of daily returns

In [ ]:
plt.figure(figsize=(10, 4))
plt.hist(portfolio_returns, bins=80, color='steelblue', edgecolor='white')
plt.axvline(portfolio_returns.mean(), color='black', linestyle='--', label='mean')
plt.title('Distribution of Daily Portfolio Returns')
plt.xlabel('Daily return')
plt.legend()
plt.tight_layout()
plt.show()

## 6. Save the portfolio series
These daily value / P&L / return series are the direct input to the VaR engine in later phases.

In [ ]:
result.to_csv('Portfolio_Construction.csv')
print('Saved Portfolio_Construction.csv with columns:', list(result.columns))
result.tail()

## Summary
- Built an **equal-weighted**, monthly-rebalanced portfolio starting at 1,000,000 units.
- Produced daily portfolio **value**, **P&L**, and **returns**.
- Reported headline stats (total return, CAGR, volatility, Sharpe, max drawdown) and charts.
- Saved `Portfolio_Construction.csv` for the VaR / backtesting phases.

This completes Tasks 1–3 (Phases 1–3) of the Market Risk project.

*Submitted by Atharv Pratap Singh Chauhan.*